In [80]:
import os
import findspark

os.environ["SPARK_HOME"] = "/workspaces/Stream_Analytics-Group_7/spark-4.0.1-bin-hadoop3"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

findspark.init()

import pyspark
print("PySpark version:", pyspark.__version__)
print("SPARK_HOME:", os.environ["SPARK_HOME"])
print("JAVA_HOME:", os.environ["JAVA_HOME"])

PySpark version: 4.0.1
SPARK_HOME: /workspaces/Stream_Analytics-Group_7/spark-4.0.1-bin-hadoop3
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [81]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("group7-gold-outputs")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-azure:3.3.4,com.microsoft.azure:azure-storage:8.6.6"
    )
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.1


In [82]:
import os
from pathlib import Path
from dotenv import load_dotenv

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd

env_path = repo_root / ".env"
load_dotenv(env_path, override=True)

print("Repo root:", repo_root)
print("ENV exists:", env_path.exists())
print("STORAGE_ACCOUNT_NAME:", os.getenv("STORAGE_ACCOUNT_NAME"))
print("STORAGE_CONTAINER_NAME:", os.getenv("STORAGE_CONTAINER_NAME"))
print("STORAGE_ACCOUNT_ACCESS_KEY loaded:", bool(os.getenv("STORAGE_ACCOUNT_ACCESS_KEY")))

Repo root: /workspaces/Stream_Analytics-Group_7
ENV exists: True
STORAGE_ACCOUNT_NAME: iesstsabbadbaa
STORAGE_CONTAINER_NAME: group7
STORAGE_ACCOUNT_ACCESS_KEY loaded: True


In [83]:
bronze_orders_path = repo_root / "data" / "bronze" / "orders"
bronze_delivery_path = repo_root / "data" / "bronze" / "delivery"

orders = spark.read.parquet(str(bronze_orders_path))
delivery = spark.read.parquet(str(bronze_delivery_path))

print("Orders:", orders.count())
print("Delivery:", delivery.count())

Orders: 94
Delivery: 738


In [84]:
from pyspark.sql.functions import (
    col,
    when,
    count,
    countDistinct,
    sum,
    avg,
    lit,
    window,
    unix_timestamp,
    expr
)

In [85]:
orders_silver = (
    orders
    .select(
        "event_id", "order_id", "event_type", "restaurant_id", "zone_id",
        "courier_id", "event_time", "ingest_time", "order_value",
        "is_promo", "estimated_delivery_minutes", "version"
    )
    .filter(col("event_id").isNotNull())
    .filter(col("order_id").isNotNull())
    .filter(col("zone_id").isNotNull())
    .filter(col("event_time").isNotNull())
)

orders_silver.show(5, truncate=False)

+------------------------------------+------------------------------------+----------------+-------------+-------+----------+-----------------------+-----------------------+-----------+--------+--------------------------+-------+
|event_id                            |order_id                            |event_type      |restaurant_id|zone_id|courier_id|event_time             |ingest_time            |order_value|is_promo|estimated_delivery_minutes|version|
+------------------------------------+------------------------------------+----------------+-------------+-------+----------+-----------------------+-----------------------+-----------+--------+--------------------------+-------+
|64dce1a6-2081-43d3-a08e-37e1a9b9aa4d|f750a99f-fa2c-45e0-b0c3-1da8d120fb87|prep_started    |rest_20      |east   |NULL      |2026-04-16 20:19:24.093|2026-04-16 20:36:46.102|37.25      |false   |36                        |1      |
|6dc3ffd9-5d22-43a3-b084-c78e20d54106|f750a99f-fa2c-45e0-b0c3-1da8d120fb87|picke

In [86]:
delivery_silver = (
    delivery
    .select(
        "event_id", "courier_id", "event_type", "courier_status", "zone_id",
        "order_id", "event_time", "ingest_time", "latitude", "longitude", "version"
    )
    .filter(col("event_id").isNotNull())
    .filter(col("courier_id").isNotNull())
    .filter(col("zone_id").isNotNull())
    .filter(col("event_time").isNotNull())
)

delivery_silver.show(5, truncate=False)

+------------------------------------+----------+---------------+---------------------+-------+------------------------------------+-----------------------+-----------------------+---------+---------+-------+
|event_id                            |courier_id|event_type     |courier_status       |zone_id|order_id                            |event_time             |ingest_time            |latitude |longitude|version|
+------------------------------------+----------+---------------+---------------------+-------+------------------------------------+-----------------------+-----------------------+---------+---------+-------+
|076e8d68-8be1-47e8-bcf6-c893b137cb42|courier_0 |status_change  |idle                 |south  |NULL                                |2026-03-06 12:26:03.427|2026-03-06 12:26:03.427|0.0      |0.0      |1      |
|440cf909-3515-4dc0-b0a7-af47eb8a9ad7|courier_0 |status_change  |waiting_at_restaurant|south  |11e87881-98f0-4e81-b0c0-9feb400e21c9|2026-03-06 12:26:33.427|2026-03-

In [87]:
gold_orders_kpi_5m = (
    orders_silver
    .groupBy(window("event_time", "5 minutes"), "zone_id")
    .agg(
        count("*").alias("orders_total"),
        countDistinct("order_id").alias("unique_orders"),
        sum(when(col("event_type") == "cancelled", 1).otherwise(0)).alias("cancelled_orders"),
        sum(when(col("event_type") == "delivered", 1).otherwise(0)).alias("delivered_orders"),
        avg("order_value").alias("avg_order_value"),
        sum(when(col("event_type") == "delivered", col("order_value")).otherwise(0)).alias("total_revenue")
    )
    .withColumn(
        "cancel_rate",
        col("cancelled_orders") / (col("orders_total") + lit(1))
    )
)

gold_orders_kpi_5m.printSchema()
gold_orders_kpi_5m.show(truncate=False)

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- orders_total: long (nullable = false)
 |-- unique_orders: long (nullable = false)
 |-- cancelled_orders: long (nullable = true)
 |-- delivered_orders: long (nullable = true)
 |-- avg_order_value: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- cancel_rate: double (nullable = true)

+------------------------------------------+-------+------------+-------------+----------------+----------------+------------------+------------------+-----------+
|window                                    |zone_id|orders_total|unique_orders|cancelled_orders|delivered_orders|avg_order_value   |total_revenue     |cancel_rate|
+------------------------------------------+-------+------------+-------------+----------------+----------------+------------------+------------------+-----------+
|{2026-04-16 20:15:00,

In [88]:
from pyspark.sql.functions import countDistinct
waiting_orders = (
    orders_silver
    .filter(col("event_type") == "created")
    .groupBy("zone_id")
    .agg(countDistinct("order_id").alias("waiting_orders"))
)

available_delivery = (
    delivery_silver
    .filter(col("event_type") == "status_change")
    .groupBy("zone_id")
    .agg(countDistinct("courier_id").alias("available_couriers"))
)

gold_demand_supply_zone = (
    waiting_orders
    .join(available_delivery, on="zone_id", how="outer")
    .fillna(0)
    .withColumn(
        "demand_supply_ratio",
        col("waiting_orders") / (col("available_couriers") + lit(1))
    )
)

gold_demand_supply_zone.show(truncate=False)

+-------+--------------+------------------+-------------------+
|zone_id|waiting_orders|available_couriers|demand_supply_ratio|
+-------+--------------+------------------+-------------------+
|center |8             |10                |0.7272727272727273 |
|east   |3             |15                |0.1875             |
|north  |0             |13                |0.0                |
|south  |2             |9                 |0.2                |
|west   |5             |13                |0.35714285714285715|
+-------+--------------+------------------+-------------------+



In [89]:
orders_created = (
    orders_silver
    .filter(col("event_type") == "created")
    .select(col("order_id"), col("zone_id"), col("event_time").alias("created_time"))
)

orders_delivered = (
    orders_silver
    .filter(col("event_type") == "delivered")
    .select(col("order_id"), col("event_time").alias("delivered_time"))
)

delivery_time_per_order = (
    orders_created
    .join(orders_delivered, on="order_id", how="inner")
    .withColumn(
        "delivery_minutes",
        (unix_timestamp("delivered_time") - unix_timestamp("created_time")) / 60.0
    )
    .filter(col("delivery_minutes") >= 0)
    .filter(col("delivery_minutes") <= 180)
)

zone_delivery_stats = (
    delivery_time_per_order
    .groupBy("zone_id")
    .agg(
        avg("delivery_minutes").alias("avg_delivery_minutes"),
        count("*").alias("delivered_orders"),
        expr("percentile_approx(delivery_minutes, 0.95)").alias("p95_delivery_minutes")
    )
)

gold_delivery_anomalies = (
    zone_delivery_stats
    .withColumn(
        "anomaly_flag",
        when((col("avg_delivery_minutes") > 45) | (col("p95_delivery_minutes") > 60), 1).otherwise(0)
    )
)

gold_delivery_anomalies.show(truncate=False)

+-------+--------------------+----------------+--------------------+------------+
|zone_id|avg_delivery_minutes|delivered_orders|p95_delivery_minutes|anomaly_flag|
+-------+--------------------+----------------+--------------------+------------+
|east   |31.666666666666668  |3               |37.0                |0           |
|west   |36.666666666666664  |3               |46.0                |0           |
|center |39.57142857142857   |7               |45.0                |0           |
|south  |35.5                |2               |40.0                |0           |
+-------+--------------------+----------------+--------------------+------------+



In [90]:
delivery_events_count = delivery_silver.count()
active_zones_count = orders_silver.select("zone_id").distinct().count()

gold_system_kpis = spark.createDataFrame(
    [(
        int(gold_orders_kpi_5m.agg(sum("orders_total")).collect()[0][0] or 0),
        int(delivery_events_count),
        int((gold_orders_kpi_5m.agg(sum("orders_total")).collect()[0][0] or 0) + delivery_events_count),
        int(active_zones_count)
    )],
    ["order_events", "delivery_events", "total_events", "active_zones"]
)

gold_system_kpis.show(truncate=False)

+------------+---------------+------------+------------+
|order_events|delivery_events|total_events|active_zones|
+------------+---------------+------------+------------+
|94          |738            |832         |4           |
+------------+---------------+------------+------------+



In [91]:
gold_business_kpis = spark.createDataFrame(
    [(
        int(orders_silver.select("order_id").distinct().count()),
        int(orders_silver.filter(col("event_type") == "delivered").select("order_id").distinct().count()),
        int(orders_silver.filter(col("event_type") == "cancelled").select("order_id").distinct().count()),
        float(
            orders_silver
            .filter(col("event_type") == "delivered")
            .agg(sum("order_value"))
            .collect()[0][0] or 0.0
        )
    )],
    ["unique_orders", "delivered_orders", "cancelled_orders", "total_revenue"]
)

gold_business_kpis = gold_business_kpis.withColumn(
    "cancel_rate",
    col("cancelled_orders") / (col("unique_orders") + lit(1))
)

gold_business_kpis.show(truncate=False)

+-------------+----------------+----------------+------------------+-------------------+
|unique_orders|delivered_orders|cancelled_orders|total_revenue     |cancel_rate        |
+-------------+----------------+----------------+------------------+-------------------+
|18           |15              |3               |347.45000000000005|0.15789473684210525|
+-------------+----------------+----------------+------------------+-------------------+



In [92]:
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
storage_container_name = os.getenv("STORAGE_CONTAINER_NAME")
storage_account_key = os.getenv("STORAGE_ACCOUNT_ACCESS_KEY")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)

blob_base = f"wasbs://{storage_container_name}@{storage_account_name}.blob.core.windows.net"

gold_orders_blob_path = f"{blob_base}/gold/orders_kpi_5m"
gold_demand_supply_blob_path = f"{blob_base}/gold/demand_supply_zone"
gold_anomalies_blob_path = f"{blob_base}/gold/delivery_anomalies"

print(blob_base)
print(gold_orders_blob_path)
print(gold_demand_supply_blob_path)
print(gold_anomalies_blob_path)

wasbs://group7@iesstsabbadbaa.blob.core.windows.net
wasbs://group7@iesstsabbadbaa.blob.core.windows.net/gold/orders_kpi_5m
wasbs://group7@iesstsabbadbaa.blob.core.windows.net/gold/demand_supply_zone
wasbs://group7@iesstsabbadbaa.blob.core.windows.net/gold/delivery_anomalies


In [93]:
gold_orders_kpi_5m.write.mode("overwrite").parquet(gold_orders_blob_path)
gold_demand_supply_zone.write.mode("overwrite").parquet(gold_demand_supply_blob_path)
gold_delivery_anomalies.write.mode("overwrite").parquet(gold_anomalies_blob_path)

print("Gold outputs written to Azure Blob")

26/04/19 18:12:17 WARN AzureFileSystemThreadPoolExecutor: Disabling threads for Delete operation as thread count 0 is <= 1
26/04/19 18:12:18 WARN AzureFileSystemThreadPoolExecutor: Disabling threads for Rename operation as thread count 0 is <= 1
26/04/19 18:12:18 WARN AzureFileSystemThreadPoolExecutor: Disabling threads for Delete operation as thread count 0 is <= 1
26/04/19 18:12:19 WARN AzureFileSystemThreadPoolExecutor: Disabling threads for Delete operation as thread count 0 is <= 1
26/04/19 18:12:20 WARN AzureFileSystemThreadPoolExecutor: Disabling threads for Rename operation as thread count 0 is <= 1
26/04/19 18:12:21 WARN AzureFileSystemThreadPoolExecutor: Disabling threads for Delete operation as thread count 0 is <= 1
26/04/19 18:12:22 WARN AzureFileSystemThreadPoolExecutor: Disabling threads for Delete operation as thread count 0 is <= 1
26/04/19 18:12:23 WARN AzureFileSystemThreadPoolExecutor: Disabling threads for Rename operation as thread count 0 is <= 1
26/04/19 18:12:2

Gold outputs written to Azure Blob


In [94]:
blob_orders = spark.read.parquet(gold_orders_blob_path)
blob_demand_supply = spark.read.parquet(gold_demand_supply_blob_path)
blob_anomalies = spark.read.parquet(gold_anomalies_blob_path)

print("Blob orders rows:", blob_orders.count())
print("Blob demand_supply rows:", blob_demand_supply.count())
print("Blob anomalies rows:", blob_anomalies.count())

blob_orders.show(5, truncate=False)
blob_demand_supply.show(5, truncate=False)
blob_anomalies.show(5, truncate=False)

Blob orders rows: 25
Blob demand_supply rows: 5
Blob anomalies rows: 4
+------------------------------------------+-------+------------+-------------+----------------+----------------+------------------+-------------+-----------+
|window                                    |zone_id|orders_total|unique_orders|cancelled_orders|delivered_orders|avg_order_value   |total_revenue|cancel_rate|
+------------------------------------------+-------+------------+-------------+----------------+----------------+------------------+-------------+-----------+
|{2026-04-16 20:15:00, 2026-04-16 20:20:00}|west   |10          |5            |0               |0               |23.822            |0.0          |0.0        |
|{2026-04-16 20:35:00, 2026-04-16 20:40:00}|center |13          |6            |0               |0               |28.016153846153845|0.0          |0.0        |
|{2026-04-16 20:25:00, 2026-04-16 20:30:00}|south  |3           |2            |0               |0               |21.723333333333333|0.

In [95]:
from pathlib import Path

gold_local_root = repo_root / "data" / "gold"
gold_local_root.mkdir(parents=True, exist_ok=True)

gold_local_orders_path = gold_local_root / "orders_kpi_5m"
gold_local_demand_supply_path = gold_local_root / "demand_supply_zone"
gold_local_anomalies_path = gold_local_root / "delivery_anomalies"
gold_local_system_kpis_path = gold_local_root / "system_kpis"
gold_local_business_kpis_path = gold_local_root / "business_kpis"

print(gold_local_root)

/workspaces/Stream_Analytics-Group_7/data/gold


In [96]:
from pyspark.sql.functions import col

gold_orders_kpi_dashboard = gold_orders_kpi_5m.select(
    col("window.start").alias("window_start"),
    col("window.end").alias("window_end"),
    col("zone_id"),
    col("orders_total"),
    col("unique_orders"),
    col("cancelled_orders"),
    col("delivered_orders"),
    col("avg_order_value"),
    col("total_revenue"),
    col("cancel_rate")
)

gold_orders_kpi_dashboard.show(truncate=False)

+-------------------+-------------------+-------+------------+-------------+----------------+----------------+------------------+------------------+-----------+
|window_start       |window_end         |zone_id|orders_total|unique_orders|cancelled_orders|delivered_orders|avg_order_value   |total_revenue     |cancel_rate|
+-------------------+-------------------+-------+------------+-------------+----------------+----------------+------------------+------------------+-----------+
|2026-04-16 20:15:00|2026-04-16 20:20:00|west   |10          |5            |0               |0               |23.822            |0.0               |0.0        |
|2026-04-16 20:35:00|2026-04-16 20:40:00|center |13          |6            |0               |0               |28.016153846153845|0.0               |0.0        |
|2026-04-16 20:25:00|2026-04-16 20:30:00|south  |3           |2            |0               |0               |21.723333333333333|0.0               |0.0        |
|2026-04-16 20:55:00|2026-04-16 21

In [97]:
from pyspark.sql.functions import countDistinct
countDistinct("order_id").alias("unique_orders"),

(Column<'count(order_id) AS unique_orders'>,)

In [98]:
gold_orders_kpi_dashboard.write.mode("overwrite").parquet(str(gold_local_orders_path))
gold_demand_supply_zone.write.mode("overwrite").parquet(str(gold_local_demand_supply_path))
gold_delivery_anomalies.write.mode("overwrite").parquet(str(gold_local_anomalies_path))
gold_system_kpis.write.mode("overwrite").parquet(str(gold_local_system_kpis_path))
gold_business_kpis.write.mode("overwrite").parquet(str(gold_local_business_kpis_path))

print("Local Gold outputs written")

Local Gold outputs written


In [99]:
print("orders_kpi_5m exists:", gold_local_orders_path.exists())
print("demand_supply_zone exists:", gold_local_demand_supply_path.exists())
print("delivery_anomalies exists:", gold_local_anomalies_path.exists())
print("system_kpis exists:", gold_local_system_kpis_path.exists())
print("business_kpis exists:", gold_local_business_kpis_path.exists())

orders_kpi_5m exists: True
demand_supply_zone exists: True
delivery_anomalies exists: True
system_kpis exists: True
business_kpis exists: True
